## สคริปต์ทำความสะอาดข้อมูล: renewable_energy_share_2000_2025.csv

In [1]:
import pandas as pd
import numpy as np

RAW_PATH = "renewable_energy_share_2000_2025.csv"
CONTINENT_PATH = "country-and-continent-codes-list-csv.csv"
V7_OUT = "renewable_energy_countries_selected_v7.csv"


# =====================================================================================
# STEP 0: โหลดไฟล์ต้นฉบับ
# =====================================================================================
print("=" * 80)
print("STEP 0: โหลดไฟล์ต้นฉบับ")
print("=" * 80)

raw = pd.read_csv(RAW_PATH)
continents = pd.read_csv(CONTINENT_PATH)

print(f"raw (ข้อมูลดิบทั้งหมด) shape: {raw.shape}")
print(f"continents (ตารางเทียบทวีป) shape: {continents.shape}")

STEP 0: โหลดไฟล์ต้นฉบับ
raw (ข้อมูลดิบทั้งหมด) shape: (7636, 33)
continents (ตารางเทียบทวีป) shape: (258, 6)


In [3]:
# =====================================================================================
# ขั้นที่ 1: คัดเฉพาะ "ประเทศจริง" ออกมาจากข้อมูลดิบ
# =====================================================================================

print("\n" + "=" * 80)
print("ขั้นที่ 1: คัดเฉพาะประเทศจริง (ตัดกลุ่มรวม/ภูมิภาค/กลุ่มรายได้ออก)")
print("=" * 80)

iso_code_continent = set(
    continents["Three_Letter_Country_Code"].dropna().astype(str).str.strip().str.upper()
)
iso_code_raw = raw["iso_code"].astype(str).str.strip().str.upper()

real_countries = iso_code_raw.isin(iso_code_continent) | (raw["country"] == "Kosovo") #ดึงกลับเข้ามาเพราะเป็นประเทศจริง แต่ไม่มี iso_code ในตารางทวีป

df = raw[real_countries].copy()

# ตัดประเทศที่ไม่มีข้อมูลไฟฟ้าเลยสักปปี
country_ele = df.groupby("country")["renewables_share_elec"].transform(
    lambda s: s.notna().any()
)
df = df[country_ele]

# ตัด Western Sahara ออกเพราะมีข้อมูลไฟฟ้าแค่ช่วงปี 2000-2009
df = df[~df["country"].isin(["Western Sahara"])].reset_index(drop=True)

print(f"shape หลังคัดประเทศ: {df.shape}")
print(f"จำนวนประเทศที่เหลือ: {df['country'].nunique()} ประเทศ")


ขั้นที่ 1: คัดเฉพาะประเทศจริง (ตัดกลุ่มรวม/ภูมิภาค/กลุ่มรายได้ออก)
shape หลังคัดประเทศ: (5381, 33)
จำนวนประเทศที่เหลือ: 213 ประเทศ


In [4]:
# =====================================================================================
# ขั้นที่ 2: ตัดคอลัมน์ที่ไม่ได้ใช้ออก
# =====================================================================================
print("\n" + "=" * 80)
print("ขั้นที่ 2: ตัดคอลัมน์ที่ไม่ต้องใช้ออก")
print("=" * 80)

del_col = [
    "population",
    "primary_energy_consumption",
    "electricity_demand",
    "renewables_share_energy",
    "fossil_share_energy",
    "low_carbon_share_energy",
    "biofuel_electricity",
    "energy_per_capita",
    "renewables_energy_per_capita",
    "fossil_energy_per_capita",
    "renewables_cons_change_twh",
]

df = df.drop(columns=del_col).reset_index(drop=True)
print(f"shape หลังตัดคอลัมน์: {df.shape}")


ขั้นที่ 2: ตัดคอลัมน์ที่ไม่ต้องใช้ออก
shape หลังตัดคอลัมน์: (5381, 22)


In [ ]:
# =====================================================================================
# ขั้นที่ 3: เติมค่าว่าง (NaN) ด้วย 0 เฉพาะจุดที่สมเหตุสมผล
# =====================================================================================

print("\n" + "=" * 80)
print("ขั้นที่ 3: เติมค่าว่าง (NaN) ด้วย 0 เฉพาะจุดที่เหมาะสม")
print("=" * 80)

col_fill = [
    "solar_electricity", "wind_electricity", "hydro_electricity", "nuclear_electricity",
    "coal_electricity", "gas_electricity", "oil_electricity", "other_renewable_electricity",
    "solar_share_elec", "wind_share_elec", "hydro_share_elec", "nuclear_share_elec",
    "coal_share_elec", "gas_share_elec",
]
df[col_fill] = df[col_fill].fillna(0)

col_sum = ["renewables_share_elec", "fossil_share_elec", "low_carbon_share_elec"]
condition_lesotho = (df["country"] == "Lesotho") & (df[col_sum].isna().any(axis=1))
df.loc[condition_lesotho, col_sum] = df.loc[condition_lesotho, col_sum].fillna(0)

print(f"shape หลังเติม NaN: {df.shape}")
print("\nค่าว่าง (NaN) ที่ยังเหลืออยู่ (เหลืออยู่ตามที่ตั้งใจ ไม่ใช่บั๊ก):")
missing = df.isna().sum()
for col in missing[missing > 0].index:
    print(f"  - {col}: {int(missing[col])} แถว")

In [ ]:
# =====================================================================================
# ขั้นที่ 4: เติมคอลัมน์ทวีป (continent)
# =====================================================================================

print("\n" + "=" * 80)
print("ขั้นที่ 4: เติมคอลัมน์ continent (ใช้โค้ดจริงจาก add_continent_column.ipynb)")
print("=" * 80)

continents_clean = continents.copy()
continents_clean["Three_Letter_Country_Code"] = (
    continents_clean["Three_Letter_Country_Code"].astype(str).str.strip().str.upper()
)

duplicate = continents_clean[continents_clean.duplicated("Three_Letter_Country_Code", keep=False)]
if not duplicate.empty:
    print("พบ iso_code ที่ซ้ำกันมากกว่า 1 ทวีปในตาราง continents (จะใช้ค่าที่เจอก่อนในไฟล์):")
    print(sorted(duplicate["Three_Letter_Country_Code"].unique()))

iso_continent = (
    continents_clean.drop_duplicates("Three_Letter_Country_Code", keep="first")
    .set_index("Three_Letter_Country_Code")["Continent_Name"]
    .to_dict()
)
iso_continent.pop("", None)
iso_continent.pop("NAN", None)

iso_normalized = df["iso_code"].astype(str).str.strip().str.upper()
iso_normalized = iso_normalized.replace({"": pd.NA, "NAN": pd.NA})

df["continent"] = iso_normalized.map(iso_continent)

จับคู่ไม่ได้ = df["continent"].isna()
print(f"จับคู่ทวีปได้: {(~จับคู่ไม่ได้).sum()} / {len(df)} แถว")
print(f"จับคู่ทวีปไม่ได้: {จับคู่ไม่ได้.sum()} / {len(df)} แถว")

df["continent"] = df["continent"].fillna("Unknown")

#เติมเอง
df.loc[df["country"] == "Kosovo", "continent"] = "Europe"

# จัดตำแหน่งคอลัมน์ continent ให้อยู่ถัดจาก country 
all_col = list(df.columns)
all_col.remove("continent")
ตำแหน่งแทรก = all_col.index("country") + 1
all_col.insert(ตำแหน่งแทรก, "continent")
df = df[all_col]

print(f"shape หลังเติม continent: {df.shape}")


ขั้นที่ 4: เติมคอลัมน์ continent (ใช้โค้ดจริงจาก add_continent_column.ipynb)
พบ iso_code ที่ซ้ำกันมากกว่า 1 ทวีปในตาราง continents (จะใช้ค่าที่เจอก่อนในไฟล์):
['ARM', 'AZE', 'CYP', 'GEO', 'KAZ', 'RUS', 'TUR', 'UMI']
จับคู่ทวีปได้: 5355 / 5381 แถว
จับคู่ทวีปไม่ได้: 26 / 5381 แถว
shape หลังเติม continent: (5381, 23)


In [6]:
# =====================================================================================
# ขั้นที่ 5: จัดคอลัมน์/ลำดับให้ตรงกับไฟล์ output สุดท้าย แล้วบันทึกเป็นไฟล์เดียว
# =====================================================================================

print("\n" + "=" * 80)
print("ขั้นที่ 5: จัดลำดับคอลัมน์ให้ตรงกับ output สุดท้าย แล้วบันทึกไฟล์")
print("=" * 80)

final_columns = [
    "country", "continent", "year", "iso_code", "gdp",
    "electricity_generation", "renewables_share_elec", "fossil_share_elec",
    "low_carbon_share_elec",
    "solar_electricity", "wind_electricity", "hydro_electricity", "nuclear_electricity",
    "coal_electricity", "gas_electricity", "oil_electricity", "other_renewable_electricity",
    "solar_share_elec", "wind_share_elec", "hydro_share_elec", "nuclear_share_elec",
    "coal_share_elec", "gas_share_elec",
]

df_final = df[final_columns].sort_values(["country", "year"]).reset_index(drop=True)
df_final.to_csv(V7_OUT, index=False)

print(f"df final shape: {df_final.shape}")
print(f"บันทึกไฟล์สุดท้ายเรียบร้อย (ไฟล์เดียว): {V7_OUT}")



ขั้นที่ 5: จัดลำดับคอลัมน์ให้ตรงกับ output สุดท้าย แล้วบันทึกไฟล์
df final shape: (5381, 23)
บันทึกไฟล์สุดท้ายเรียบร้อย (ไฟล์เดียว): renewable_energy_countries_selected_v7.csv
